# Hands-On Perception Project: Side Scan Sonar Loop Closure

#### Juan Castillo
#### Max Puig
#### David Villanueva

# 1) Define data paths 

In [21]:
import torch 
import cv2 
import os 
import numpy as np 
from pathlib import Path

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') # Define GPU device

# For data preparation
input_dir = r'data'     # XTF file folder
output_dir = r'output'  # Output dataset folder
segment_size = 2000
overlap_size = 1000
upper_limit = 2 ** 15

# For inference
test_image_path = r'output/images'
test_range_path = r'output/range'
test_altitude_path = r'output/altitude'
weight_path = r'best_model_v610_eagle.pth'     # Select the best weights for inference
# weight_path = r'best_model_v610_jaguar.pth'
# weight_path = r'best_model_v611_jaguar.pth'
output_inf = r'output/visual_test' # Result saving path

# For terrain mask generation
altitude_dir = r"output/altitude"
range_dir = r"output/range"
depth_dir = r"output/visual_test/z_gray" # Predicted height map
height_output_dir = r"output/visual_test/height_visual" # Store the grayscale image of the generated height map and the mask of its understated terrain areas.
output_folder = r"output/visual_test/z_mask"

# 2) Run PhysDepth 

## 2.1) Prepare data

##### Takes the xtf files and separates the waterfall into different images according to the segment size and overlap size. Stores the images, range, altitude and shadow in the output folder, as well as blind masks for both channels

In [3]:
# The images are saved into the output folder
from physdnet.prep import prepare_data
prepare_data(input_dir, output_dir, segment_size, overlap_size, upper_limit, 'both')
prepare_data(input_dir, output_dir, segment_size, overlap_size, upper_limit, 'left')
prepare_data(input_dir, output_dir, segment_size, overlap_size, upper_limit, 'right')

Processing file: 2025-09-24_09-25-24_0.xtf
	Number of blind bins: 680
Mask calculated!
mask_left: (2000, 2000), mask_right: (2000, 2000)
left channel data shape (72675, 2000)
left channel data shape 145350000
2025-09-24_09-25-24_0.xtf: total_rows=72675, segments=72, stride=1000
Processing file: 2025-09-24_09-25-24_0.xtf
	Number of blind bins: 680
Mask calculated!
mask_left: (2000, 2000), mask_right: (2000, 2000)
left channel data shape (72675, 2000)
left channel data shape 145350000
2025-09-24_09-25-24_0.xtf: total_rows=72675, segments=72, stride=1000
Processing file: 2025-09-24_09-25-24_0.xtf
	Number of blind bins: 680
Mask calculated!
mask_left: (2000, 2000), mask_right: (2000, 2000)
left channel data shape (72675, 2000)
left channel data shape 145350000
2025-09-24_09-25-24_0.xtf: total_rows=72675, segments=72, stride=1000


## 2.2) Run inference

##### This generates the inference for z, z_gray (elevation normal and grayed), rho, rho_gray (range normal and grayed), path loss, theta, phi and shadow in the output folder

In [3]:
from physdnet.test import run_inference
run_inference(device, test_image_path, test_range_path, test_altitude_path, weight_path, output_inf, 'left')
run_inference(device, test_image_path, test_range_path, test_altitude_path, weight_path, output_inf, 'right')
run_inference(device, test_image_path, test_range_path, test_altitude_path, weight_path, output_inf, 'both')

Image saved to: output_left/visual_test/theta/2025-09-24_09-25-24_0.xtf_left_069_theta.png
Image saved to: output_left/visual_test/z/2025-09-24_09-25-24_0.xtf_left_069_z.png
Image saved to: output_left/visual_test/path/2025-09-24_09-25-24_0.xtf_left_069_path.png
Image saved to: output_left/visual_test/rho/2025-09-24_09-25-24_0.xtf_left_069_rho.png
Image saved to: output_left/visual_test/theta/2025-09-24_09-25-24_0.xtf_left_070_theta.png
Image saved to: output_left/visual_test/z/2025-09-24_09-25-24_0.xtf_left_070_z.png
Image saved to: output_left/visual_test/path/2025-09-24_09-25-24_0.xtf_left_070_path.png
Image saved to: output_left/visual_test/rho/2025-09-24_09-25-24_0.xtf_left_070_rho.png
Image saved to: output_left/visual_test/theta/2025-09-24_09-25-24_0.xtf_left_044_theta.png
Image saved to: output_left/visual_test/z/2025-09-24_09-25-24_0.xtf_left_044_z.png
Image saved to: output_left/visual_test/path/2025-09-24_09-25-24_0.xtf_left_044_path.png
Image saved to: output_left/visual_te

In [7]:
# Both channels
total_images = len(os.listdir(test_image_path))
print(f"Total test images for both channels: {total_images}")

# Left channel
total_images = len(os.listdir(test_image_path.replace("output", "output_left")))
print(f"Total test images for left channel: {total_images}")

# Right channel
total_images = len(os.listdir(test_image_path.replace("output", "output_right")))
print(f"Total test images for right channel: {total_images}")

Total test images for both channels: 144
Total test images for left channel: 72
Total test images for right channel: 72


## 2.3) Terrain masks

#### Generates masks for elevation and shadow for feature filtering and saves the result in height_visual and z_mask


In [8]:
from physdnet.terrain_mask import terrain_mask
terrain_mask(altitude_dir, range_dir, depth_dir, height_output_dir, output_folder, "left")
terrain_mask(altitude_dir, range_dir, depth_dir, height_output_dir, output_folder, "right")
terrain_mask(altitude_dir, range_dir, depth_dir, height_output_dir, output_folder, "both")

100%|██████████| 72/72 [00:00<00:00, 120.20it/s]


Processing complete


100%|██████████| 72/72 [00:00<00:00, 117.84it/s]


Processing complete


100%|██████████| 144/144 [00:01<00:00, 118.21it/s]


Processing complete


## 3) Run MINIMA on the left side for feature matching and straction

### Combine masks for outlier rejection, use shadow map, blind zone, and height map

In [22]:
def combine_masks(mask1_path, mask2_path, mask3_path, output_path):
    mask1 = cv2.imread(mask1_path, cv2.IMREAD_GRAYSCALE)
    mask2 = cv2.imread(mask2_path, cv2.IMREAD_GRAYSCALE)
    mask3 = cv2.imread(mask3_path, cv2.IMREAD_GRAYSCALE)

    print(f"mask 1 shape: {mask1.shape}, mask 2 shape: {mask2.shape}, mask 3 shape: {mask3.shape}")

    combined_mask = ((mask1 > 0) & (mask2 > 0) & (mask3 > 0)).astype(np.uint8) * 255

    cv2.imwrite(output_path, combined_mask)

def filter_matches_with_masks(kpts0, kpts1, mask0, mask1):
    '''Applies a mask to a set of keypoints'''
    def prepare_mask(mask: torch.Tensor, device):
        mask = mask.to(device)

        if mask.dim() == 3:
            # if RGB, take one channel
            mask = mask[0]

        # binary: 1 invalid, 0 valid
        mask = (mask > 0.5).to(torch.uint8)
        return mask
    # Gets the device
    device = kpts0.device

    # Threshold the masks from int to bool
    mask0 = prepare_mask(mask0, device)
    mask1 = prepare_mask(mask1, device)

    # Gets the mask dimensions
    H0, W0 = mask0.shape
    H1, W1 = mask1.shape

    # Extract the keypoints coordinates
    x0i = torch.round(kpts0[:, 0]).long()
    y0i = torch.round(kpts0[:, 1]).long()
    x1i = torch.round(kpts1[:, 0]).long()
    y1i = torch.round(kpts1[:, 1]).long()

    inside0 = (x0i >= 0) & (x0i < W0) & (y0i >= 0) & (y0i < H0)
    inside1 = (x1i >= 0) & (x1i < W1) & (y1i >= 0) & (y1i < H1)

    valid0 = torch.zeros_like(inside0, dtype=torch.bool)
    valid1 = torch.zeros_like(inside1, dtype=torch.bool)

    valid0[inside0] = (mask0[y0i[inside0], x0i[inside0]] == 0)
    valid1[inside1] = (mask1[y1i[inside1], x1i[inside1]] == 0)

    keep = valid0 & valid1
    return kpts0[keep], kpts1[keep]

# For the left side only
shadow_mask_path =  r"output_left/shadow"
z_mask_path =  r"output_left/shadow"
blind_mask = Path(r"output_left/2025-09-24_09-25-24_0.xtf_mask_left.png")

# Get all the files, they should be the same number, one per image
shadow_mask_files = [f for f in os.listdir(shadow_mask_path) if f.lower().endswith('.png')]
z_mask_files = [f for f in os.listdir(z_mask_path) if f.lower().endswith('.png')]

# Generate all the masks by combining the shadow mask, z mask, and blind mask
masks_output = r"output_left/final_masks"
os.makedirs(masks_output, exist_ok=True)
i = 0
for shadow_file, z_mask_file in zip(shadow_mask_files, z_mask_files):
    shadow_i_path = os.path.join(shadow_mask_path, shadow_file)
    z_i_path = os.path.join(z_mask_path, z_mask_file)
    output_i = os.path.join(masks_output, f"mask_{i}.png")
    combine_masks(shadow_i_path, z_i_path, blind_mask, output_i)
    i += 1

mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask 1 shape: (2000, 2000), mask 2 shape: (2000, 2000), mask 3 shape: (2000, 2000)
mask

In [ ]:
from minima_pipeline import MinimaMatcher, run_pipeline_on_images

# Build matcher for in-memory images
matcher = MinimaMatcher(method="sp_lg", ckpt="./weights/minima_lightglue.pth", use_path=False)

img0 = cv2.imread("image_a.png")
img1 = cv2.imread("image_b.png")

# Your binary masks, same size as each image
mask0 = cv2.imread("mask_a.png", cv2.IMREAD_GRAYSCALE)
mask1 = cv2.imread("mask_b.png", cv2.IMREAD_GRAYSCALE)

result = run_pipeline_on_images(
    matcher,
    img0,
    img1,
    mask0=mask0,
    mask1=mask1,
    ransac_reproj_threshold=3.0,
)

print("Raw matches:", len(result.mkpts0))
print("After mask filtering:", len(result.mkpts0_kept))
print("RANSAC inliers:", 0 if result.ransac_inliers is None else int(result.ransac_inliers.sum()))
print("Homography:\n", result.H)